# 📝 LangChain 에이전트와 도구 과제 LV2(응용) — RAG 체인·검색 평가·SQL·MCP·외부 API

> LV1 에서 익힌 도구·궤적·평가 지표를 **조합**합니다. RAG 를 체인으로 조립하고, 검색기를 숫자로 재어 K 를 고르며, 데이터베이스·MCP·외부 API 를 도구로 연결합니다.

## 풀이 방법
1. 맨 위 **준비 셀들**(제공 코드)을 위에서부터 실행하세요. `.env` 에 본인 **`OPENAI_API_KEY`** 가 필요합니다. **데이터베이스는 준비물이 없습니다** — SQL 단원에서 배운 sqlite 를 쓰므로 접속 정보가 필요 없습니다.
2. 모델이 만든 답과 도구 선택은 **실행할 때마다 달라집니다** — 채점은 **타입·구조**로 합니다.
3. **2·3번**은 모델을 부르지 않는 **검색 평가**라 값이 언제나 같습니다 — 여기서는 숫자로 채점합니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요(임베딩 모델 로딩에 잠시 걸립니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 서점 FAQ 문서를 LangChain 부품으로 색인합니다(임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다).
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

faq_df = pd.read_csv('data/bookstore_faq.csv')

# 표의 한 행 = Document 하나. page_content 는 검색 대상 본문, metadata 는 함께 붙일 꼬리표입니다.
faq_docs = [Document(page_content=row.text, metadata={'id': row.id, 'title': row.title})
            for row in faq_df.itertuples()]

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델

# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
faq_store = Chroma.from_documents(faq_docs, embeddings, collection_name='bookstore_faq',
                                  ids=faq_df['id'].tolist())
faq_retriever = faq_store.as_retriever(search_kwargs={'k': 2})   # 질문마다 가장 가까운 2개를 돌려주는 검색기

print('색인 완료 — 문서 수:', len(faq_docs))

In [ ]:
# [제공 코드] 에이전트 공통 준비
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool          # 7번에서 도구를 직접 만들 때 씁니다

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. RAG 체인 직접 조립하기
**배경**: 교안에서 본 것처럼, 도구도 에이전트도 없이 **검색 → 프롬프트 → 모델 → 문자열**을 한 줄로 잇는 **체인**을 직접 조립합니다. 맨 위 준비 셀이 만들어 둔 **`faq_retriever`** 를 씁니다.

**요구사항**: 두 가지를 만드세요.

**(1) 기본 RAG 체인 `rag_chain`**

- `ChatPromptTemplate` 로 프롬프트를 만드세요. **`{context}`** 와 **`{question}`** 두 자리를 두고, *주어진 자료에 있는 내용만으로 한국어로 답하라*는 지시를 담습니다.
- 체인은 **검색 결과를 글로 합친 것**을 `context` 에, **질문 그대로**를 `question` 에 채운 뒤 프롬프트 → 모델 → 출력파서를 차례로 잇습니다. 질문을 그대로 흘려보내는 데에는 **`RunnablePassthrough`**, 최종 결과를 문자열로 받는 데에는 **`StrOutputParser`** 를 씁니다.
- 검색 결과를 한 덩어리 글로 합치는 **`format_docs`** 는 아래 제공 셀에 있습니다.
- 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`answer1`** 에 담으세요(문자열입니다).

**(2) 근거를 함께 돌려주는 체인 `chain_with_sources`**

- 실무에서 **출처 없는 RAG 답**은 쓰기 어렵습니다. **`RunnableParallel`** 로 두 갈래를 묶어 답과 근거를 함께 받으세요 — **`'answer'`** 에는 (1)의 체인을, **`'sources'`** 에는 검색기를 그대로 둡니다.
- 같은 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`result1`** 에 담으세요. `result1['answer']` 는 문자열, `result1['sources']` 는 **`Document` 목록**이 됩니다.

**예시**: `answer1` 은 전자책 기기 수를 설명하는 한국어 문장입니다(**문장은 실행할 때마다 달라지므로** 채점은 **타입·구조**로만 합니다). `result1['sources']` 의 각 `Document` 에는 `metadata['title']` 이 붙어 있어 어느 FAQ 에서 왔는지 알 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 딕셔너리로 두 자리를 채우고, 파이프로 프롬프트·모델·파서를 잇는다.

세부구현:
1. ChatPromptTemplate 로 자료와 질문 자리를 가진 프롬프트를 만든다.
   1-1. 자료에 있는 내용만으로 답하라는 지시를 넣는다.
2. context 자리에는 검색기와 format_docs 를 이어 붙인 것을, question 자리에는
   입력을 그대로 흘려보내는 부품을 둔다.
3. 그 딕셔너리에 프롬프트·모델·문자열 출력파서를 차례로 이어 체인을 만든다.
4. 답과 근거를 함께 받는 체인은 두 갈래를 나란히 묶는 부품으로 만든다.
   4-1. 한 갈래는 앞에서 만든 체인, 다른 갈래는 검색기 그대로다.
5. 두 체인에 같은 질문 문자열을 넣어 결과를 각각 담는다.
```

</details>

In [ ]:
# [제공 코드] 검색 결과(Document 목록)를 프롬프트에 넣을 한 덩어리 글로 합칩니다 — 실행만 하세요.
def format_docs(docs):
    """검색된 Document 들을 제목과 함께 한 덩어리 글로 합친다."""
    return '\n\n'.join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)


print('준비 완료 —', format_docs(faq_retriever.invoke('전자책'))[:40], '...')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 실호출이라 문장은 매번 다르다 — 타입과 구조만 본다
assert isinstance(answer1, str) and len(answer1.strip()) > 0
assert isinstance(result1, dict) and set(result1) == {'answer', 'sources'}, \
    "두 갈래의 이름은 'answer' 와 'sources' 입니다"
assert isinstance(result1['answer'], str) and len(result1['answer'].strip()) > 0
assert isinstance(result1['sources'], list) and len(result1['sources']) >= 1
assert all(isinstance(d, Document) for d in result1['sources']), 'sources 는 Document 목록입니다'
assert all('title' in d.metadata for d in result1['sources'])
# sources 가 정말 검색기에서 왔는지 — 같은 질문을 검색기에 직접 넣어 대조한다(검색은 결정적이다)
want1 = [d.metadata['id'] for d in faq_retriever.invoke('전자책은 몇 대의 기기에서 볼 수 있나요?')]
assert [d.metadata['id'] for d in result1['sources']] == want1, \
    'sources 갈래에 검색기를 그대로 두세요'
print('✅ 통과!')

## 2. 검색 품질 재기 — 평가셋 전체를 네 지표로
**배경**: 지금까지 검색 도구는 **눈으로 훑어** 그럴듯한지 봤습니다. 이제 **숫자로** 잽니다. 아래 평가셋에는 질문 25개마다 **정답 조각 id** 가 붙어 있습니다. 문항마다 검색을 하고 LV1 에서 만든 네 지표를 계산해 평균을 내면 이 검색기의 **성적표**가 됩니다.

> 이 문제의 코퍼스는 앞 문제들의 서점 FAQ 가 아니라 **개인정보 질의응답 모음집**입니다. 아래 **제공 셀 세 개**(평가 코퍼스 살펴보기 → 청킹·색인 → 지표 함수)를 위에서부터 실행한 뒤 푸세요. 검색은 **`search_ids(질문, k)`** 로 하고, 조각 id 를 1위부터 순서대로 돌려줍니다.

**요구사항**: `K = 3` 으로 평가셋 전체를 재서 세 가지를 만드세요.

- 문항마다 상위 3개 조각 id 를 얻습니다. 검색은 문항당 **한 번만** 하고, 그 결과 하나로 네 지표를 모두 계산합니다.
- 정답 라벨 `gold_chunks` 는 `'|'` 로 이어져 있으니 **나눠서 리스트로** 만들어 넘깁니다.
- 문항별 결과를 DataFrame **`eval_results`** 로 만드세요. 열은 **`query_id`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**여야 합니다. 한 행이 한 문항이므로 행 수는 평가 문항 수와 같습니다.
- 네 지표 각각의 평균을 딕셔너리 **`eval_summary`** 에 담으세요. 열쇠는 **`'Hit'`, `'P'`, `'R'`, `'MRR'`** 네 개입니다.
- `Hit` 이 `0.0` 인 문항의 `query_id` 를 리스트 **`missed2`** 에 담으세요(평가셋에 나온 순서 그대로).

**예시**: 제대로 재면 평균은 **Hit 약 0.96 · P 약 0.53 · R 약 0.73 · MRR 약 0.92** 근처가 나오고, `missed2` 에는 **문항 하나**만 남습니다 — 어느 질문이 걸렸는지 직접 열어 보세요. 환경에 따라 값이 조금 다를 수 있어 채점은 **넉넉한 범위**로 봅니다(값을 정확히 맞힐 필요는 없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 문항을 한 번씩 돌면서 검색을 한 번만 하고, 그 결과 목록으로 네 지표를 모두 계산한다.

세부구현:
1. 빈 목록을 만들어 두고 평가셋을 한 행씩 돈다.
   1-1. 질문으로 상위 3개 조각 id 를 얻는다.
   1-2. 정답 라벨을 구분자로 나눠 목록으로 만든다.
   1-3. 네 지표를 계산해 딕셔너리 하나로 담아 목록에 넣는다(적은 순서가 곧 열 순서다).
2. 목록으로 DataFrame 을 만든다.
3. 네 열의 평균을 딕셔너리로 만든다.
4. Hit 이 0 인 행만 골라 그 문항 번호를 목록으로 뽑는다.
```

</details>

In [ ]:
# [제공 코드] 평가 코퍼스와 평가셋을 읽고 눈으로 확인합니다 — 이 셀은 실행만 하세요.
import pandas as pd

# 문서 모음 — 한 행이 문서 하나이고, 검색 대상 글은 '본문' 열에 있습니다.
qna_df = pd.read_csv('data/qna_docs.csv')
# 평가셋 — 질문(query)마다 정답 조각 id(gold_chunks)가 '|' 로 이어져 붙어 있습니다.
evalset = pd.read_csv('data/qna_eval_chunk.csv')

print('문서', len(qna_df), '건 / 평가 문항', len(evalset), '건')
display(qna_df[['id', '분야', '질문']].head(3))
display(evalset[['query_id', 'query', 'gold_chunks', '유형']].head(3))

In [ ]:
# [제공 코드] 청킹 — RAG 파이프라인을 만든 단원의 규칙 그대로입니다(문단을 모아 size 근처에서 끊습니다).
def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모아 size 근처에서 끊는다(문단 자체는 쪼개지 않는다)."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# [제공 코드] 문서를 조각으로 나눠 색인하고 검색 함수를 만듭니다 — 이 셀은 실행만 하세요.
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 조각 id 는 '문서id-순번' 규칙입니다. 평가셋의 정답 라벨이 이 규칙으로 붙어 있으므로
#  청킹 기준(400)이나 id 규칙을 바꾸면 정답과 어긋나 점수가 전부 달라집니다.
qna_chunks = []
for doc_id, text in zip(qna_df['id'], qna_df['본문']):
    for i, part in enumerate(chunk_paragraph(text, 400)):
        qna_chunks.append(Document(page_content=part, metadata={'chunk_id': f'{doc_id}-{i}'}))

# 임베딩은 맨 위 준비 셀에서 만든 것을 그대로 씁니다(같은 모델을 두 번 불러올 이유가 없습니다).
# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 조각이 중복되지 않습니다.
qna_store = Chroma.from_documents(qna_chunks, embeddings, collection_name='qna_eval',
                                  ids=[d.metadata['chunk_id'] for d in qna_chunks])


def search_ids(query, k):
    """질문과 의미가 가까운 조각 id 를 1위부터 k개까지 순서대로 돌려준다."""
    # 평가에서는 k 를 바꿔 가며 재야 해서, k 를 인자로 받는 검색을 그대로 씁니다.
    return [d.metadata['chunk_id'] for d in qna_store.similarity_search(query, k=k)]


print('조각', len(qna_chunks), '개 색인 완료')
print('첫 문항 검색 결과:', search_ids(evalset['query'].iloc[0], 3))

In [ ]:
# [제공 코드] LV1 에서 만든 네 지표 — 여기서는 다시 만들지 말고 그대로 씁니다.
def hit_at_k(ranked, gold, k):
    """상위 k개 안에 정답이 하나라도 있으면 1.0, 하나도 없으면 0.0 을 돌려준다."""
    return 1.0 if any(r in gold for r in ranked[:k]) else 0.0


def precision_at_k(ranked, gold, k):
    """상위 k개 중 정답의 비율을 돌려준다(나누는 수는 언제나 k)."""
    return sum(1 for r in ranked[:k] if r in gold) / k


def recall_at_k(ranked, gold, k):
    """그 질문의 전체 정답 중 상위 k개로 건진 비율을 돌려준다."""
    return sum(1 for r in ranked[:k] if r in gold) / len(gold)


def mrr_at_k(ranked, gold, k):
    """처음 만난 정답의 순위 역수를 돌려준다(상위 k개 안에 없으면 0.0)."""
    for rank, r in enumerate(ranked[:k], 1):
        if r in gold:
            return 1 / rank
    return 0.0


print('지표 준비 완료 — hit_at_k / precision_at_k / recall_at_k / mrr_at_k')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 값을 손으로 적어 넣으면 통과하지 못하도록, 채점이 검색을 다시 돌려 정답을 스스로 계산한다
# (임베딩·검색은 결정적이라 같은 질문·같은 k 면 결과가 항상 같다)
assert list(eval_results.columns) == ['query_id', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert len(eval_results) == len(evalset), '모든 문항을 재야 합니다'

for query_id, query, gold_text in zip(evalset['query_id'], evalset['query'], evalset['gold_chunks']):
    ranked = search_ids(query, 3)
    gold = gold_text.split('|')
    mine = eval_results[eval_results['query_id'] == query_id].iloc[0]
    assert mine['Hit'] == hit_at_k(ranked, gold, 3), f'{query_id} Hit'
    assert abs(mine['P'] - precision_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} P'
    assert abs(mine['R'] - recall_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} R'
    assert abs(mine['MRR'] - mrr_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} MRR'

# 평균은 값을 정확히 맞히는 대신 '이 언저리인지'만 본다
#  (임베딩 모델 버전이 달라 한두 문항이 뒤집혀도 옳은 풀이는 통과하고,
#   분모를 잘못 쓴 계산은 다른 지표 값으로 넘어가 버리므로 범위 밖으로 떨어진다)
for metric, low, high in [('Hit', 0.85, 1.00), ('P', 0.45, 0.65),
                          ('R', 0.65, 0.85), ('MRR', 0.85, 1.00)]:
    assert abs(eval_summary[metric] - eval_results[metric].mean()) < 1e-9, f'eval_summary[{metric}] 가 다릅니다'
    assert low < eval_summary[metric] <= high, f'{metric} 평균이 지문에 적힌 언저리를 벗어났습니다'

assert missed2 == eval_results.loc[eval_results['Hit'] == 0.0, 'query_id'].tolist()
assert len(missed2) == 1, '상위 3개 안에 정답이 하나도 없는 문항은 한 건입니다'
print('✅ 통과!')

## 3. K 를 바꿔 가며 비교 — 넓게 볼수록 좋을까
**배경**: 2번에서는 K 를 3 으로 고정했습니다. K 는 **우리가 정하는 손잡이**입니다 — 크게 잡으면 많이 꺼내 오고, 작게 잡으면 조금만 꺼내 옵니다. 어느 쪽이 나은지는 **재 봐야** 압니다.

**요구사항**: `K = 1, 3, 5, 10` 네 가지로 각각 평가셋 전체를 재서 DataFrame **`k_table`** 을 만드세요.

- 열은 **`K`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**입니다. 한 행이 K 하나이고, 행 순서는 **1, 3, 5, 10** 입니다.
- 각 칸은 그 K 로 잰 **전 문항 평균**입니다(2번과 같은 방법, K 만 바꿉니다).
- 검색은 문항마다 **가장 큰 K(=10)로 한 번만** 하고, 그 목록 하나로 네 K 를 모두 계산하세요. 지표 함수가 알아서 앞에서 `k` 개만 보므로 목록을 다시 자를 필요가 없습니다.

**예시**: 위의 두 행은 대략 이렇게 나옵니다(소수 셋째 자리까지 — 환경에 따라 조금 다를 수 있어 채점은 **넉넉한 범위와 방향**으로 봅니다).

```
 K    Hit      P      R    MRR
 1  0.880  0.880  0.451  0.880
 3  0.960  0.533  0.727  0.920
 5    ...    ...    ...    ...
10    ...    ...    ...    ...
```

표를 다 채우면 **K 를 키울 때 오르는 지표와 내려가는 지표가 갈립니다.** 그 모습을 확인하고 아래 서술 답안을 채우세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 상위 10개를 한 번 받아 두고, 같은 목록에 k 만 바꿔 넣어 네 번 잰다.

세부구현:
1. 평가셋을 한 번 돌며 (상위 10개 조각 id, 정답 목록) 짝을 모아 둔다.
2. K 후보 네 개를 차례로 돈다.
   2-1. 모아 둔 짝마다 네 지표를 계산해 문항 수로 나눠 평균을 낸다.
   2-2. K 와 네 평균을 딕셔너리 하나로 담아 목록에 넣는다.
3. 목록으로 DataFrame 을 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert k_table['K'].tolist() == [1, 3, 5, 10], 'K 는 1, 3, 5, 10 순서여야 합니다'

# 채점도 같은 방식으로 다시 재서 대조한다 — 표에 값을 적어 넣는 것으로는 통과하지 못한다
pairs = [(search_ids(query, 10), gold_text.split('|'))
         for query, gold_text in zip(evalset['query'], evalset['gold_chunks'])]
for i, k in enumerate([1, 3, 5, 10]):
    for metric, metric_fn in [('Hit', hit_at_k), ('P', precision_at_k),
                              ('R', recall_at_k), ('MRR', mrr_at_k)]:
        want = sum(metric_fn(r, g, k) for r, g in pairs) / len(pairs)
        assert abs(k_table.loc[i, metric] - want) < 1e-9, f'K={k} 의 {metric} 값이 다릅니다'

# 지문에 적어 둔 K=1 · K=3 행이 그 언저리인지 본다(정확한 값이 아니라 범위로)
assert 0.80 < k_table.loc[0, 'Hit'] <= 1.00, 'K=1 의 Hit 가 지문의 언저리를 벗어났습니다'
assert 0.35 < k_table.loc[0, 'R'] < 0.55, 'K=1 의 Recall 이 지문의 언저리를 벗어났습니다'
assert 0.45 < k_table.loc[1, 'P'] < 0.65, 'K=3 의 Precision 이 지문의 언저리를 벗어났습니다'

# K 를 키우면 Recall 은 오르고 Precision 은 떨어진다 -- 맞바꿈이 표에 그대로 보인다
assert k_table.loc[3, 'R'] > k_table.loc[0, 'R'], 'K 가 커지면 Recall 은 올라야 합니다'
assert k_table.loc[3, 'P'] < k_table.loc[0, 'P'], 'K 가 커지면 Precision 은 내려야 합니다'
print('✅ 통과!')

**서술 답안** — 표를 보고 아래에 적으세요.

*(여기에 이 검색기의 K 를 얼마로 정할지, 표의 어떤 값을 근거로 그렇게 정했는지 서술하세요)*

## 4. Text-to-SQL — 주문 건수 조회, 그리고 답을 스키마로
**배경**: 자연어 질문을 SQL 로 바꿔 답하게 합니다. **SELECT 전용** 도구와 **스키마 안내 프롬프트**를 함께 씁니다. 그다음, 같은 질문의 답을 **문장이 아니라 데이터**로 받아 봅니다.

**요구사항**: 두 가지를 만드세요.

**(1) 궤적 읽기**

- 아래 준비 셀들(`run_select` 도구·`SCHEMA_PROMPT`·데이터베이스·`OrderAnswer` 스키마)을 먼저 실행하세요.
- `create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT)` 로 에이전트를 만들고, 질문 **"c1 고객이 주문한 건수는 모두 몇 건인가요?"** 로 `invoke` 한 결과를 변수 **`res4`** 에 담으세요.
- 궤적에서 **ToolMessage** 만 골라 변수 **`tool_msgs4`** 에 담으세요.

**(2) 도구도 쓰고, 답도 스키마로 — `response_format`**

- (1)의 최종 답은 **자유 문장**이라 그대로는 표에 넣지 못합니다. 교안 01 3절에서 본 것처럼 `create_agent` 에 **`response_format=ProviderStrategy(OrderAnswer, strict=True)`** 를 더해 에이전트를 하나 더 만드세요(임포트는 `from langchain.agents.structured_output import ProviderStrategy`).
- **같은 질문** "c1 고객이 주문한 건수는 모두 몇 건인가요?" 로 `invoke` 한 결과를 변수 **`res4b`** 에 담고, 거기서 정형 결과를 꺼내 변수 **`report4`** 에 담으세요. 정형 결과는 결과 딕셔너리의 **`'structured_response'`** 열쇠에 들어 있습니다.

**예시**: c1 고객의 주문은 3건이라 (1)의 도구 결과 어딘가에 **'3'** 이 들어 있습니다(모델이 표를 먼저 둘러보느라 조회를 여러 번 할 수도 있습니다). (2)의 `report4` 는 `OrderAnswer` 객체이고, `report4.order_count` 는 **3**, `report4.sql` 에는 실행한 **`select`** 문이, `report4.answer` 에는 사용자에게 보여 줄 한국어 한 문장이 들어 있습니다. 문장 내용은 실행마다 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 스키마 프롬프트를 준 SQL 에이전트로 질문을 실행하고, 도구 결과를 읽는다.
- 답까지 정형화하려면 에이전트를 만들 때 인자를 하나 더 준다 - 도구 목록은 그대로다.

세부구현:
1. create_agent 에 [run_select] 와 system_prompt=SCHEMA_PROMPT 를 준다.
2. 질문을 그대로 invoke 해 res4 에 담는다.
3. ToolMessage 만 골라 tool_msgs4 에 담는다.
4. 같은 인자에 response_format 을 더해 두 번째 에이전트를 만들고 같은 질문을 invoke 한다.
   4-1. 결과 딕셔너리에 키가 하나 더 생겨 있다 - 궤적은 그대로 남는다.
5. 그 키에서 정형 결과를 꺼내 report4 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 데이터베이스 준비 — SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'bookstore.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('data/setup_day19.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 — 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 —', DB_PATH)

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 — 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 — 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 — 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

In [ ]:
# [제공 코드] Text-to-SQL 용 스키마 안내 프롬프트
SCHEMA_PROMPT = (
    '너는 온라인 서점 데이터베이스 조회를 돕는 도우미다. run_select 도구로 SELECT 문만 실행해 답하라. 표 스키마는 다음과 같다. bs_customer(customer_id, name, grade, city): 고객. bs_book(book_id, title, author, genre, price, stock): 도서. bs_order(order_id, customer_id, book_id, quantity, order_date): 주문. 조회 결과를 바탕으로 한국어로 간단히 답하라.'
)

In [ ]:
# [제공 코드] 4번 (2)에서 쓸 결과 스키마 — 이 셀은 실행만 하세요.
from pydantic import BaseModel, Field


class OrderAnswer(BaseModel):
    """주문 건수 문의 한 건을 처리한 결과."""

    sql: str = Field(description="조회에 사용한 SELECT 문")
    # 이 칸은 모델이 지어내는 값이 아니라 '도구가 알려 준 값'이다.
    order_count: int = Field(description="도구가 알려 준 주문 건수")
    answer: str = Field(description="사용자에게 보여 줄 한국어 한 문장")


print("스키마 준비 완료")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert tool_msgs4 == [m for m in res4['messages'] if isinstance(m, ToolMessage)]
assert res4['messages'][0].text.strip() == 'c1 고객이 주문한 건수는 모두 몇 건인가요?', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs4) >= 1, 'SQL 도구가 한 번도 불리지 않았습니다'
assert any('3' in str(m.content) for m in tool_msgs4)   # c1 의 주문 건수 3
assert isinstance(res4['messages'][-1].text, str)

# (2) 정형 결과 — 값이 아니라 '모양'을 스키마가 보장한다
assert report4 is res4b['structured_response'], 'report4 는 res4b 에서 꺼낸 정형 결과여야 합니다'
assert isinstance(report4, OrderAnswer)
# response_format 을 줘도 도구는 그대로 불린다 — 궤적이 남아 있는지 본다
assert any(isinstance(m, ToolMessage) for m in res4b['messages']), \
    'response_format 을 줘도 도구 궤적은 남습니다 — 도구가 불리지 않았습니다'
assert report4.order_count == 3, 'order_count 는 도구가 알려 준 주문 건수(3)여야 합니다'
assert 'select' in report4.sql.lower(), 'sql 칸에는 조회에 쓴 SELECT 문이 들어갑니다'
assert isinstance(report4.answer, str) and report4.answer.strip()
print('✅ 통과!')

## 5. MCP 도구 연결
**배경**: 표준 규격(MCP)으로 만든 외부 도구를 에이전트에 붙입니다. `data/mcp_server.py` 에 환율·날짜 도구가 있습니다.

**요구사항**:
- 아래 준비 셀(비동기 준비·MCP 클라이언트)을 실행해 `mcp_tools` 를 받으세요.
- `create_agent(model, mcp_tools)` 로 에이전트를 만들고, 질문 **"9100원은 몇 달러인가요?"** 로 `agent.ainvoke(...)` 를 `asyncio.run(...)` 으로 감싸 실행한 결과를 변수 **`res5`** 에 담으세요.
- 궤적에서 **ToolMessage** 만 골라 변수 **`tool_msgs5`** 에 담으세요.

**예시**: 불린 도구 이름은 `krw_to_usd`, 결과 문자열에는 **'7.0'** 이 들어 있습니다(9100 ÷ 1300 — 서버가 고정 환율로 계산하므로 값 자체는 도구가 정합니다).

<details><summary>힌트</summary>

```text
접근방법:
- 받아 온 mcp_tools 로 에이전트를 만들고, 비동기 실행을 asyncio.run 으로 감싼다.

세부구현:
1. create_agent 에 mcp_tools 를 넘긴다.
2. asyncio.run(agent.ainvoke({...})) 로 질문을 실행해 res5 에 담는다.
3. ToolMessage 만 골라 tool_msgs5 에 담는다(MCP 결과는 리스트일 수 있으니 str 로 확인).
```

</details>

In [ ]:
# [제공 코드] 비동기 실행 준비 — MCP 도구는 비동기(async)라 이 셀이 필요합니다(실행만 하세요).
import asyncio

import nest_asyncio

nest_asyncio.apply()   # Jupyter 안에서도 asyncio.run(...) 을 쓸 수 있게 해 준다


In [ ]:
# [제공 코드] MCP 서버를 서브프로세스로 띄우고 도구 목록을 가져옵니다.
import sys
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

# transport='stdio' : 서버를 별도 프로세스로 띄우고 표준입출력으로 이야기한다는 뜻입니다.
server_path = str((Path('data') if Path('data').exists() else Path('../data')) / 'mcp_server.py')
mcp_client = MultiServerMCPClient({
    'utils': {'transport': 'stdio', 'command': sys.executable, 'args': [server_path]},
})
mcp_tools = asyncio.run(mcp_client.get_tools())   # 서버가 어떤 도구를 갖고 있는지 물어봅니다
print('MCP 도구:', [t.name for t in mcp_tools])

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert tool_msgs5 == [m for m in res5['messages'] if isinstance(m, ToolMessage)]
assert res5['messages'][0].text.strip() == '9100원은 몇 달러인가요?', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs5) >= 1, 'MCP 도구가 한 번도 불리지 않았습니다'
assert 'krw_to_usd' in [m.name for m in tool_msgs5]
assert any('7.0' in str(m.content) for m in tool_msgs5)   # 9100 / 1300 = 7.0
print('✅ 통과!')

## 6. MCP 검색 → 구조화 수집
**배경**: MCP `search_new_books` 도구는 신간을 **줄글 텍스트**로 돌려줍니다. 이를 `BookList` 스키마로 구조화해 **표(DataFrame)로 적재**하는 "수집 → 정형화" 파이프라인을 만듭니다(교안의 구조화된 출력 응용).

아래 **제공 셀**의 스키마(`BookList`)와 지시문(`COLLECT_INSTR`)을 씁니다.

**요구사항**:
- 5번에서 받은 `mcp_tools` 중 이름이 **`search_new_books`** 인 도구를 골라, **`keyword`** 인자에 **`'소설'`** 을 넣어 부르고 그 결과를 변수 **`raw6`** 에 담으세요. MCP 도구는 5번처럼 **비동기**(`ainvoke`)라 `asyncio.run(...)` 으로 감싸야 합니다 — 이번에는 에이전트를 거치지 않고 **도구를 직접** 부릅니다.
- MCP 결과는 조각 리스트일 수 있으니 **텍스트만** 모아 변수 **`found6`** 에 담으세요(제공 셀의 `mcp_text` 사용).
- `model.with_structured_output(BookList)` 로 **`COLLECT_INSTR + found6`** 를 `invoke` 해 변수 **`books6`** 에 담으세요.
- `books6.books` 를 `pandas.DataFrame` 으로 만들어 변수 **`df6`** 에 담으세요(각 `BookInfo` 는 `.model_dump()`).

**예시**: '소설' 로 검색하면 신간 2권이 나오고, `df6` 은 `title·author·price·summary` 열을 가진 표가 됩니다 (열 이름은 스키마가 정하므로 고정, 행 수는 모델이 몇 권을 담느냐에 달려 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- MCP 도구로 검색 -> 텍스트만 추출 -> BookList 로 구조화 -> DataFrame 으로 적재.

세부구현:
1. mcp_tools 에서 이름이 search_new_books 인 도구 하나를 골라 낸다.
2. 그 도구를 keyword 인자로 비동기 호출하고 asyncio.run 으로 감싸 raw6 에 담는다.
3. 제공 셀의 mcp_text 로 조각에서 텍스트만 모아 found6 에 담는다.
4. 모델에 BookList 스키마를 씌우고, 수집 지시문과 found6 을 이어 붙여 넘긴다.
5. 결과의 books 를 하나씩 딕셔너리로 바꿔 DataFrame 으로 만든다.
```

</details>

In [ ]:
# [제공 코드] 신간 수집용 스키마 — 이 셀은 실행만 하세요.
from pydantic import BaseModel, Field


class BookInfo(BaseModel):
    """검색으로 찾은 도서 한 권의 정보."""

    title: str = Field(description="책 제목")
    author: str = Field(description="지은이")
    price: int = Field(description="가격(원)")
    summary: str = Field(description="한 줄 소개")


class BookList(BaseModel):
    """여러 권의 도서 정보 묶음."""

    books: list[BookInfo] = Field(description="검색으로 찾은 도서 목록")


print("스키마 준비 완료")

In [ ]:
# [제공 코드] 수집 지시문·도우미 — 이 셀은 실행만 하세요.
import pandas as pd

COLLECT_INSTR = '다음 검색 결과를 도서 목록으로 정리하세요:\n'


def mcp_text(raw):
    """MCP 도구 결과에서 텍스트만 모은다(조각마다 붙는 임시 id 는 실행마다 바뀌므로 버린다)."""
    return raw if isinstance(raw, str) else '\n'.join(b['text'] for b in raw)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# found6 이 MCP 검색에서 온 글인지부터 본다 — 서버가 '소설' 로 돌려주는 두 권은 늘 같다
assert isinstance(found6, str) and len(found6) > 0
assert '밤의 서점 이야기' in found6 and '바다 건너 우체국' in found6, \
    "MCP 도구를 '소설' 로 검색한 결과를 텍스트로 모아 found6 에 담으세요"

assert list(df6.columns) == ['title', 'author', 'price', 'summary']
assert len(df6) >= 1
assert df6['price'].dtype.kind in 'iu'   # 가격은 정수(스키마가 int 로 못박았다)
assert df6['title'].map(type).eq(str).all()
# 표를 손으로 지어내도 위 조건은 다 만족한다 — df6 이 books6 을 옮긴 것인지 대조한다
assert len(df6) == len(books6.books), 'books6 의 책 수와 df6 의 행 수가 다릅니다'
assert set(df6['title']) == {b.title for b in books6.books}, 'books6 을 그대로 옮긴 표가 아닙니다'
print('✅ 통과!')

## 7. 외부 API 를 도구로 — 오늘의 엔화 환율
**배경**: 5번의 MCP `krw_to_usd` 는 **1300원 고정**이었습니다. 오늘의 값이 필요하면 **인터넷에 직접 물어봐야** 합니다. 이번엔 달러가 아니라 **엔화**로, 그리고 **금액을 인자로 받는** 도구를 직접 만듭니다.

`https://api.frankfurter.dev/v1/latest` 에 `amount`·`from`·`to` 를 붙여 요청하면 환산 결과가 옵니다 (키가 필요 없는 공개 API). 예를 들어 `{'amount': 100, 'from': 'JPY', 'to': 'KRW'}` 로 요청하면 응답 JSON 의 `rates['KRW']` 에 **100엔에 해당하는 원화**가 들어 있습니다.

> **인터넷은 언제든 끊깁니다.** 교안 01 4절의 **도구 설계 원칙 3 — 실패는 예외가 아니라 문자열로 돌려준다** 를 이 도구에 적용하세요. 예외를 밖으로 던지면 에이전트가 그 자리에서 멈추지만, 문자열이면 모델이 그것을 읽고 사용자에게 안내하거나 다시 시도할 수 있습니다.

**요구사항**:
- `@tool` 을 붙인 함수 **`jpy_krw_rate(amount_jpy: int) -> str`** 를 정의하세요. docstring 을 한국어로 적습니다.
- `import requests` 한 뒤 함수 안에서 **`requests.get(...)`** 으로 위 주소에 요청하세요. `params={'amount': amount_jpy, 'from': 'JPY', 'to': 'KRW'}` 와 **`timeout=10`** 을 줍니다.
- 요청과 `raise_for_status()`·`.json()` 을 **`try` 안에** 두고, **`except requests.RequestException`** 으로 실패를 잡으세요. 잡았을 때는 **예외를 다시 던지지 말고**, 왜 실패했는지 담은 **문자열**을 돌려줍니다.
- 성공했을 때 반환 문자열은 **`f'{amount_jpy}엔 = {round(원화)}원'`** 형식입니다(원화는 `rates['KRW']`, **반올림해 정수**로).
- `create_agent(model, [jpy_krw_rate])` 로 에이전트를 만들고, 질문 **"300엔은 우리 돈으로 얼마야?"** 로 `invoke` 한 결과를 변수 **`res7`** 에 담은 뒤, 불린 도구 이름 목록을 **`used7`** 에 담으세요.

**예시**: `jpy_krw_rate.invoke({'amount_jpy': 100})` → `'100엔 = 921원'` 같은 문자열(**환율은 날마다 바뀌므로 숫자는 오늘 값**입니다). 요청이 실패하면 `'환율을 가져오지 못했습니다: ...'` 처럼 **사정을 알리는 문자열**이 나옵니다. `used7` 에는 `'jpy_krw_rate'` 가 들어 있습니다.

> **채점 안내**: 자가채점은 성공 경로뿐 아니라 **실패 경로도** 확인합니다 — `requests.get` 이 실패하도록 잠깐 바꿔 놓고 도구를 불러, 예외가 밖으로 나오지 않고 **문자열**이 돌아오는지 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 데이터 수집 단원에서 배운 requests 요청과 JSON 읽기를 @tool 함수 안에 넣되, 끊길 때에 대비해 try 로 감싼다.

세부구현:
1. requests 를 임포트하고 @tool 과 타입힌트·docstring 을 단다.
2. try 안에서 params 에 amount·from·to 를 넣고 timeout 을 준 뒤,
   raise_for_status 로 실패를 확인하고 응답 JSON 을 읽는다.
3. except 로 requests 의 요청 계열 예외를 잡고, 다시 던지지 말고
   실패를 알리는 문자열을 반환한다(도구 설계 원칙 3).
4. 성공하면 응답에서 원화 값을 꺼내 반올림해 지정된 형식의 문자열로 만든다.
5. 만든 도구 하나로 에이전트를 만들고 질문을 invoke 한 뒤 ToolMessage 의 name 을 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import requests as rq

# 채점도 같은 API 에 직접 물어본다 — 값을 손으로 박아 두면 오늘 환율과 어긋나 걸린다
ref9 = rq.get('https://api.frankfurter.dev/v1/latest',
              params={'amount': 100, 'from': 'JPY', 'to': 'KRW'}, timeout=10).json()
got9 = jpy_krw_rate.invoke({'amount_jpy': 100})
assert isinstance(got9, str)
assert '100엔' in got9
assert f"{round(ref9['rates']['KRW'])}원" in got9, '오늘 환율로 환산한 값이 보이지 않습니다'

# 실패 경로도 본다 — 요청이 실패하도록 requests.get 을 잠깐 바꿔 놓고 도구를 부른다
saved_get = rq.get


def failing_get(*args, **kwargs):
    """채점용 모의 상황 — 인터넷이 끊긴 것처럼 요청을 실패시킨다."""
    raise rq.RequestException('연결 실패(채점용 모의 상황)')


rq.get = failing_get
try:
    fail9 = jpy_krw_rate.invoke({'amount_jpy': 100})
finally:
    rq.get = saved_get                     # 무슨 일이 있어도 원래대로 되돌린다
assert isinstance(fail9, str), '실패해도 예외를 던지지 말고 문자열을 돌려주세요(도구 설계 원칙 3)'

assert used7 == [m.name for m in res7['messages'] if isinstance(m, ToolMessage)]
assert res7['messages'][0].text.strip() == '300엔은 우리 돈으로 얼마야?', '지문의 질문을 그대로 넣어 실행하세요'
assert 'jpy_krw_rate' in used7, '에이전트가 도구를 부르지 않았습니다'
print('✅ 통과!')

---
수고했어요! RAG 를 **체인으로 직접 조립**해 근거까지 함께 받았고, 검색기의 품질을 **숫자로 재어** K 를 고르는 근거를 만들었습니다. 데이터베이스를 도구로 붙여 **자연어 질문을 SQL 로** 바꾸고 그 답까지 **스키마에 담아** 받았으며, MCP 도구를 연결해 검색 결과를 **구조화 데이터**로 적재했습니다. 마지막으로 외부 API 를 도구로 감싸며 **실패를 문자열로 돌려주는** 습관까지 손에 익혔습니다. LV3 에서는 이것들을 묶어 **리뷰 인텔리전스 파이프라인**을 만들고, 검색기를 재서 **쓸 `k` 를 근거 있게** 고릅니다.